# End-to-End Sentiment Analysis Pipeline (Google Colab Ready)

This is a self-contained, production-ready Jupyter Notebook for **Multi-Class Sentiment Analysis** (`positive`, `negative`, `neutral`) using the **Yelp Reviews Parquet Dataset** (6.99 Million reviews).

### Pipeline Highlights:
- **Dependencies & Setup**: Automatic package installation (`pyarrow`, `pandas`, `scikit-learn`, `matplotlib`, `seaborn`).
- **Colab / Local Drive Mount**: Easy file path configuration for Google Drive or local Parquet execution.
- **Data Streaming**: Memory-efficient stratified sampling from 2.89 GB Parquet datasets.
- **Text Preprocessing**: Custom text cleaner (HTML removal, URL stripping, lowercase normalization).
- **Feature Extraction**: TF-IDF Sublinear N-gram Vectorization.
- **Model Benchmarking**: Automated training & evaluation of **Logistic Regression**, **SGD Classifier (Linear SVM)**, and **Multinomial Naive Bayes**.
- **Evaluation Metrics**: Macro/Weighted F1, Precision, Recall, and Confusion Matrix Heatmaps.
- **Model Export & Live Predictor**: Serializes trained model pipeline and provides an interactive inference engine.

--- 
## Step 1: Install Required Dependencies
Run this cell on Google Colab or local environment to ensure all necessary libraries are installed.

In [ ]:
# Install dependencies (if missing in Colab environment)
!pip install --upgrade pyarrow pandas scikit-learn matplotlib seaborn joblib

--- 
## Step 2: Import Libraries & Environment Setup

In [ ]:
import os
import re
import time
from typing import Dict, Any, List, Tuple

import pyarrow.parquet as pq
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score
)

# Set plotting style
plt.style.use('ggplot')
sns.set_theme(style="whitegrid")
print("Libraries imported successfully!")

--- 
## Step 3: Google Drive Mount & Parquet Dataset Path Setup
If running on Google Colab, uncomment the Drive mount lines below or upload the Parquet file directly to Colab.

In [ ]:
# Optional: Uncomment if dataset is in Google Drive
# from google.colab import drive
# drive.mount('/content/drive')

# Specify the path to your parquet file
PARQUET_FILE = 'part-00000-c3f61153-e06b-4f01-88ea-69e9a133a765-c000.snappy-001.parquet'

# Verify file existence
if os.path.exists(PARQUET_FILE):
    parquet_meta = pq.ParquetFile(PARQUET_FILE)
    print(f"✓ Dataset found: {PARQUET_FILE}")
    print(f"  Total Num Rows: {parquet_meta.metadata.num_rows:,}")
    print(f"  Columns:        {parquet_meta.schema.names}")
else:
    print(f"⚠️ Warning: Parquet file '{PARQUET_FILE}' not found in current directory.")
    print("  Please update 'PARQUET_FILE' variable or upload the file.")

--- 
## Step 4: Text Preprocessor & Stratified Parquet Loader

In [ ]:
class TextPreprocessor:
    """Utility class for cleaning raw review text."""
    def __init__(self, lower: bool = True, remove_html: bool = True, remove_urls: bool = True):
        self.lower = lower
        self.remove_html = remove_html
        self.remove_urls = remove_urls

    def clean_text(self, text: str) -> str:
        if not isinstance(text, str):
            return ""
        if self.remove_html:
            text = re.sub(r'<[^>]+>', ' ', text)
        if self.remove_urls:
            text = re.sub(r'http[s]?://\S+|www\.\S+', ' ', text)
        if self.lower:
            text = text.lower()
        text = re.sub(r'\s+', ' ', text).strip()
        return text

    def transform_series(self, series: pd.Series) -> pd.Series:
        return series.apply(self.clean_text)


def load_parquet_sample(
    file_path: str,
    sample_size: int = 100000,
    random_state: int = 42,
    columns: list = None
) -> pd.DataFrame:
    """Efficiently loads a stratified sample from large Parquet files."""
    if columns is None:
        columns = ['review_text', 'sentiment_label', 'stars']

    parquet_file = pq.ParquetFile(file_path)
    total_rows = parquet_file.metadata.num_rows

    if sample_size >= total_rows:
        return parquet_file.read(columns=columns).to_pandas()

    read_size = min(sample_size * 2, total_rows)
    chunks = []
    accumulated_rows = 0

    for batch in parquet_file.iter_batches(batch_size=50000, columns=columns):
        df_batch = batch.to_pandas()
        chunks.append(df_batch)
        accumulated_rows += len(df_batch)
        if accumulated_rows >= read_size:
            break

    df_full = pd.concat(chunks, ignore_index=True)
    df_full = df_full.dropna(subset=['review_text', 'sentiment_label'])

    if len(df_full) > sample_size:
        df_sampled, _ = train_test_split(
            df_full,
            train_size=sample_size,
            stratify=df_full['sentiment_label'],
            random_state=random_state
        )
        return df_sampled.reset_index(drop=True)
    
    return df_full.reset_index(drop=True)

print("Preprocessor and Data Loader components initialized.")

--- 
## Step 5: Sentiment Pipeline & Classifier Wrapper

In [ ]:
class SentimentPipeline:
    """End-to-End Sentiment Analysis TF-IDF + Machine Learning Pipeline."""
    def __init__(
        self,
        classifier_type: str = "logistic_regression",
        ngram_range: Tuple[int, int] = (1, 2),
        max_features: int = 50000,
        sublinear_tf: bool = True
    ):
        self.classifier_type = classifier_type
        self.ngram_range = ngram_range
        self.max_features = max_features
        self.sublinear_tf = sublinear_tf
        self.preprocessor = TextPreprocessor()
        self.pipeline = self._build_pipeline()

    def _get_classifier(self):
        if self.classifier_type == "logistic_regression":
            return LogisticRegression(
                C=2.0,
                max_iter=500,
                solver="lbfgs",
                random_state=42
            )
        elif self.classifier_type == "sgd":
            return SGDClassifier(
                loss="log_loss",
                penalty="l2",
                alpha=1e-5,
                max_iter=1000,
                random_state=42,
                n_jobs=-1
            )
        elif self.classifier_type == "naive_bayes":
            return MultinomialNB(alpha=0.1)
        else:
            raise ValueError(f"Unknown classifier type: {self.classifier_type}")

    def _build_pipeline(self) -> Pipeline:
        vectorizer = TfidfVectorizer(
            ngram_range=self.ngram_range,
            max_features=self.max_features,
            sublinear_tf=self.sublinear_tf,
            token_pattern=r'\b\w+\b'
        )
        classifier = self._get_classifier()
        return Pipeline([
            ('tfidf', vectorizer),
            ('clf', classifier)
        ])

    def fit(self, X: pd.Series, y: pd.Series) -> "SentimentPipeline":
        X_clean = self.preprocessor.transform_series(X)
        self.pipeline.fit(X_clean, y)
        return self

    def predict(self, X: pd.Series) -> np.ndarray:
        if isinstance(X, list):
            X = pd.Series(X)
        X_clean = self.preprocessor.transform_series(X)
        return self.pipeline.predict(X_clean)

    def predict_proba(self, X: pd.Series) -> np.ndarray:
        if isinstance(X, list):
            X = pd.Series(X)
        X_clean = self.preprocessor.transform_series(X)
        if hasattr(self.pipeline, "predict_proba"):
            return self.pipeline.predict_proba(X_clean)
        elif hasattr(self.pipeline, "decision_function"):
            decision = self.pipeline.decision_function(X_clean)
            e_x = np.exp(decision - np.max(decision, axis=1, keepdims=True))
            return e_x / e_x.sum(axis=1, keepdims=True)
        else:
            raise AttributeError("Classifier does not support probability estimation.")

    def save(self, filepath: str):
        os.makedirs(os.path.dirname(filepath), exist_ok=True)
        joblib.dump(self, filepath)
        print(f"✓ Model pipeline saved to '{filepath}'.")

    @classmethod
    def load(cls, filepath: str) -> "SentimentPipeline":
        model = joblib.load(filepath)
        print(f"✓ Model pipeline loaded from '{filepath}'.")
        return model

print("SentimentPipeline class defined successfully.")

--- 
## Step 6: Evaluator & Visualization Framework

In [ ]:
class SentimentEvaluator:
    """Evaluates multi-class classification performance & outputs metrics."""
    def __init__(self, labels: List[str] = None):
        if labels is None:
            labels = ["negative", "neutral", "positive"]
        self.labels = labels

    def evaluate(self, y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, Any]:
        acc = accuracy_score(y_true, y_pred)
        macro_f1 = f1_score(y_true, y_pred, average='macro')
        weighted_f1 = f1_score(y_true, y_pred, average='weighted')
        report_str = classification_report(y_true, y_pred, target_names=self.labels, digits=4)
        cm = confusion_matrix(y_true, y_pred, labels=self.labels)

        return {
            "accuracy": acc,
            "macro_f1": macro_f1,
            "weighted_f1": weighted_f1,
            "classification_report_str": report_str,
            "confusion_matrix": cm,
            "labels": self.labels
        }

    def plot_confusion_matrix(self, eval_results: Dict[str, Any], title: str = "Confusion Matrix"):
        """Plot seaborn confusion matrix heatmap."""
        cm = eval_results['confusion_matrix']
        labels = eval_results['labels']
        
        plt.figure(figsize=(7, 5))
        sns.heatmap(
            cm, 
            annot=True, 
            fmt="d", 
            cmap="Blues", 
            xticklabels=labels, 
            yticklabels=labels,
            cbar=False
        )
        plt.title(f"{title} (Accuracy: {eval_results['accuracy']:.2%})", fontsize=13)
        plt.xlabel("Predicted Label", fontsize=11)
        plt.ylabel("True Label", fontsize=11)
        plt.tight_layout()
        plt.show()

print("SentimentEvaluator initialized.")

--- 
## Step 7: Execute Data Loading & Model Training Benchmark
We sample **100,000 reviews**, perform a stratified 80/20 train/test split, and train/benchmark **SGD Classifier**, **Logistic Regression**, and **Multinomial Naive Bayes**.

In [ ]:
# Load sampled data
SAMPLE_SIZE = 100000
print(f"Loading {SAMPLE_SIZE:,} sampled records from Parquet dataset...")
df_sample = load_parquet_sample(PARQUET_FILE, sample_size=SAMPLE_SIZE)
print(f"Dataset Shape: {df_sample.shape}")
print("
Class Distribution:")
print(df_sample['sentiment_label'].value_counts(normalize=True).map(lambda x: f"{x:.2%}"))

# Perform Train / Test Split
X_train, X_test, y_train, y_test = train_test_split(
    df_sample['review_text'],
    df_sample['sentiment_label'],
    test_size=0.20,
    random_state=42,
    stratify=df_sample['sentiment_label']
)

print(f"
Train samples: {len(X_train):,}, Test samples: {len(X_test):,}")

# Benchmark Classifiers
evaluator = SentimentEvaluator()
candidates = ["sgd", "logistic_regression", "naive_bayes"]
model_results = {}
trained_models = {}

for clf_name in candidates:
    print(f"
{'='*50}
Training Model: {clf_name.upper()}
{'='*50}")
    t0 = time.time()
    model = SentimentPipeline(classifier_type=clf_name)
    model.fit(X_train, y_train)
    train_dur = time.time() - t0
    
    preds = model.predict(X_test)
    res = evaluator.evaluate(y_test, preds)
    res['train_time'] = train_dur
    
    model_results[clf_name] = res
    trained_models[clf_name] = model
    
    print(f"--> [{clf_name.upper()}] Time: {train_dur:.2f}s | Accuracy: {res['accuracy']:.4f} | Macro F1: {res['macro_f1']:.4f}")
    print("Classification Report:")
    print(res['classification_report_str'])
    evaluator.plot_confusion_matrix(res, title=f"{clf_name.upper()} Confusion Matrix")

--- 
## Step 8: Compare Model Benchmark Summary & Save Best Model

In [ ]:
# Benchmark Summary Table
summary_data = []
for name, res in model_results.items():
    summary_data.append({
        "Classifier": name.upper(),
        "Accuracy": f"{res['accuracy']:.2%}",
        "Macro F1": f"{res['macro_f1']:.4f}",
        "Weighted F1": f"{res['weighted_f1']:.4f}",
        "Training Time": f"{res['train_time']:.2f}s"
    })

df_summary = pd.DataFrame(summary_data)
print("===========================================================")
print("                 MODEL BENCHMARK SUMMARY")
print("===========================================================")
print(df_summary.to_string(index=False))
print("===========================================================")

# Select Best Model based on Macro F1
best_clf_name = max(model_results, key=lambda k: model_results[k]['macro_f1'])
best_model = trained_models[best_clf_name]
print(f"
--> Best overall classifier selected: {best_clf_name.upper()}")

# Save Best Model
MODEL_SAVE_PATH = os.path.join("models", "sentiment_model.pkl")
best_model.save(MODEL_SAVE_PATH)

--- 
## Step 9: Real-Time Interactive Inference Engine
Test custom review text snippets with predicted sentiment label, confidence percentage, and class probabilities.

In [ ]:
def predict_sentiment(model: SentimentPipeline, text: str):
    """Predict sentiment label and probability breakdown for input text."""
    pred_label = model.predict([text])[0]
    probas = model.predict_proba([text])[0]
    classes = model.pipeline.classes_
    
    proba_dict = {cls: float(prob) for cls, prob in zip(classes, probas)}
    confidence = max(probas)
    
    print("\n" + "=" * 60)
    print(f"Input Review Text: \"{text}\"")
    print(f"--> PREDICTED SENTIMENT: {pred_label.upper()}")
    print(f"--> CONFIDENCE SCORE:    {confidence:.2%}")
    print("-" * 60)
    print("Class Probability Breakdown:")
    for cls, prob in proba_dict.items():
        bar = "#" * int(prob * 25)
        print(f"  {cls.capitalize():<10}: {prob:6.2%} | {bar}")
    print("=" * 60)

# Load saved model artifact
loaded_sentiment_model = SentimentPipeline.load(MODEL_SAVE_PATH)

# Test Sample Inputs
test_cases = [
    "The food was absolute perfection! Staff went out of their way to make us comfortable.",
    "Extremely disappointed. Rude waiters, long wait time, and cold unseasoned food.",
    "It was an average dining experience. Decent food, standard prices, nothing outstanding."
]

for review in test_cases:
    predict_sentiment(loaded_sentiment_model, review)